# Imports

In [139]:
import pandas as pd
import numpy as np

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

import os

import sqlite3
import faiss

# Load data

In [140]:
CSV_PATH = "../data/csv/processed/"
files    = os.listdir(CSV_PATH)
csvs     = sorted([file.split('_')[1].split('.')[0] for file in files])

historic_csvs = csvs[:-1]
train_df      = pd.concat([pd.read_csv(f"{CSV_PATH}products_{csv}.csv") for csv in historic_csvs], ignore_index=True)

latest_csv = csvs[-1]
test_df    = pd.read_csv(f"{CSV_PATH}products_{latest_csv}.csv")

In [141]:
df = pd.read_csv("../data/csv/mail_groceries.csv")

conn = sqlite3.connect("../data/sql/swipes.db")
labels = pd.read_sql_query("SELECT data_id, is_liked, is_superliked, is_passed FROM swipes", conn)
conn.close()

In [142]:
labels

,data_id,is_liked,is_superliked,is_passed
0,10876682,0,0,1
1,10808108,1,0,0
2,10863250,0,1,0
3,10888400,0,0,1
4,10808240,0,0,1
...,...,...,...,...
536,10847913,0,0,1
537,10807308,0,0,1
538,10820944,1,0,0
539,10876540,1,0,0


# Pre-processing 

In [143]:
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

In [144]:
passage_train = train_df['translated_product'].to_numpy()
passage_embeddings_train = model.encode(passage_train)

passage_test = test_df['translated_product'].to_numpy()
passage_embeddings_test = model.encode(passage_test)

In [145]:
onehot = ['brand','category']
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown='ignore'), onehot),
    # ("num", StandardScaler(), ['price'])
])
X = preprocessor.fit_transform(train_df).toarray()
num_categories = sum(len(c) for c in preprocessor.named_transformers_['cat'].categories_)
print(f"Number of categories after one-hot encoding: {num_categories}")

Number of categories after one-hot encoding: 764


In [146]:
train_df_processed = pd.DataFrame(X, columns=preprocessor.get_feature_names_out())
train_df_processed = pd.merge(train_df, train_df_processed, left_index=True, right_index=True)
cat_features_train = train_df_processed.values[:,-num_categories:]
X_final_train = np.hstack((passage_embeddings_train, cat_features_train))

test_df_processed = pd.DataFrame(preprocessor.transform(test_df).toarray(), columns=preprocessor.get_feature_names_out())
test_df_processed = pd.merge(test_df, test_df_processed, left_index=True, right_index=True)
cat_features_test = test_df_processed.values[:,-num_categories:]
X_final_test = np.hstack((passage_embeddings_test, cat_features_test))

# Train/test split

In [147]:
# train, val = train_test_split(X_final_train, test_size=0.2, random_state=42)
threshold = int(0.8 * X_final_train.shape[0])
val   = X_final_train[threshold:]
train = X_final_train[:threshold]
test  = X_final_test

# Create DataFrames with embeddings + category features
embedding_cols = [f'emb_{i}' for i in range(passage_embeddings_train.shape[1])]
cat_cols = preprocessor.get_feature_names_out().tolist()
all_cols = embedding_cols + cat_cols

train_df_split = pd.DataFrame(train,columns=all_cols)
val_df_split   = pd.DataFrame(val,  columns=all_cols)
test_df_split  = pd.DataFrame(test, columns=all_cols)

# Merge with original metadata (data_id, product_name, etc.)
train_indices = train_df_processed.index[~train_df_processed.index.isin(val_df_split.index)].tolist()
val_indices   = train_df_processed.index[ train_df_processed.index.isin(val_df_split.index)].tolist()

train = pd.concat([train_df_processed.iloc[train_indices].reset_index(drop=True), train_df_split], axis=1)
val   = pd.concat([train_df_processed.iloc[val_indices  ].reset_index(drop=True), val_df_split]  , axis=1)
test  = pd.concat([test_df_processed.reset_index(drop=True), test_df_split], axis=1)

train = pd.merge(train, labels, on='data_id', how='inner')
val   = pd.merge(val,   labels, on='data_id', how='inner')

In [148]:
train.shape, val.shape, test.shape

((464, 2570), (109, 2570), (153, 2567))

# Inference

In [149]:
Xtrain = train[all_cols].to_numpy()
ytrain = train['is_liked'].to_numpy()
Xval   = val[all_cols].to_numpy()
yval   = val['is_liked'].to_numpy()

### FAISS

In [150]:
# Build FAISS index for cosine similarity over translated product embeddings
train_vecs = np.ascontiguousarray(train_df_split.astype('float32'))
test_vecs  = np.ascontiguousarray(test_df_split.astype('float32'))

# Normalize for cosine similarity (dot-product index)
faiss.normalize_L2(train_vecs)
faiss.normalize_L2(test_vecs)

index = faiss.IndexFlatIP(train_vecs.shape[1])
index.add(train_vecs)
print(f"Indexed {index.ntotal} product vectors")

# Example lookup: top-5 nearest neighbors for first test item
D, I = index.search(test_vecs[100].reshape(1,-1), 5)
print("Top-5 neighbors (indices):", I[0])
print("Similarity scores:", D[0])

Indexed 1888 product vectors
Top-5 neighbors (indices): [1072 1517 1579 1003 1576]
Similarity scores: [1.         0.6666666  0.58235776 0.5730605  0.56953543]


In [151]:
test_df.iloc[100].values

array([np.int64(10888481), np.float64(20.0), 'Klovborg', 'Skiveost',
       'Skiveost Danbo Mellemlagret', np.int64(1), np.int64(0), 'pk.',
       'Netto',
       'https://static.tilbudsugen.dk/1st-retail/2025/51/64237/Netto522025-1_13_70x1051_2823x2736_zoom.jpg',
       '2025-12-20', '2025-12-23',
       'https://res.cloudinary.com/dfqzmnlga/image/upload/v1766229456/2025-12-20/10888481.jpg',
       'Skiveost Danbo Intermediate stock',
       'Skiveost Danbo Intermediate stock: The ultimate multitasker who can flip burgers, brew coffee, and never miss a deadline. 🍔☕⏰'],
      dtype=object)

In [152]:
train_df.iloc[1576].values

array([np.int64(10847954), np.float64(25.0), 'Løgismose', 'Skiveost',
       'Skiveost Aged Havarti', np.int64(1), np.int64(0), 'pk.', 'Netto',
       '2025-11-29', '2025-12-05',
       'https://res.cloudinary.com/dfqzmnlga/image/upload/v1764415614/2025-11-29/10847954.jpg',
       'Skiveost Aged Havarti',
       'Savory, slightly cheesy, and always ready to melt into a perfect bite 🧀',
       'https://static.tilbudsugen.dk/1st-retail/2025/49/64080/Netto492025-1_11_1692x5187_3332x6870_zoom.jpg'],
      dtype=object)

### KNN

In [156]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn.fit(Xtrain, ytrain)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'cosine'
,metric_params,None
,n_jobs,None


In [157]:
display(val[:3])
knn.predict_proba(Xval[:3])

,data_id,price,brand,category,product_name,units,quantity,unit_type,store_name,start_date,...,cat__category_Vitaminer,cat__category_Vodka,"cat__category_Våben, udklædning, rollespil",cat__category_Whisky,cat__category_Øl,cat__category_Øvrige mejeriprodukter,cat__category_Øvrige vinlande,is_liked,is_superliked,is_passed
0,10808524,9.0,Shine,Rengøringsartikler,Rengøringsservietter,1,48,pk.,Netto,2025-11-08,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1
1,10808451,10.0,NaN,Papir,Servietter,1,20,pk.,Netto,2025-11-08,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1
2,10808508,79.0,Neophos,Opvask,Maskinopvask-tabs,1,40,pk.,Netto,2025-11-08,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1


array([[0.2, 0.8],
       [1. , 0. ],
       [0.8, 0.2]])

In [158]:
# get neighbors' product names according to knn
D, I = knn.kneighbors(Xval[:3])
for i, neighbors in enumerate(I):
    print(train.iloc[neighbors]['translated_product'].values)
    print(D[i])
    print()


['Chorizo' 'Potato sausage' 'Peanut butter Crunchy' 'Skimming Lost'
 'Wok mixture']
[0.44020551 0.44020553 0.64347944 0.64391151 0.64920867]

['Candy strips Strawberry' 'Carrots' 'Makeup' 'Tulles' 'Cushion covers']
[0.42628784 0.44068688 0.44068688 0.44964773 0.45207995]

['Machine-washing losses' 'Cherry sauce' 'Pillows' 'Chocolate Plate Light'
 'Pillows']
[0.47935531 0.6        0.6        0.6        0.6       ]

